# Evaluating Text-to-SQL with Prediction Powered Inference on Spider

## What this notebook demonstrates

Evaluating an AI system rigorously requires ground-truth labels — which are expensive to collect at scale.
This notebook shows how to use **Stratified Prediction Powered Inference (PPI++)** with the **GLIDE** library to get
a statistically valid accuracy estimate while dramatically reducing annotation cost.

The dataset is built from [Spider 1.0](https://huggingface.co/datasets/xlangai/spider), a widely-used text-to-SQL benchmark.
We evaluate a small Claude model's ability to translate natural language questions into SQL.

### The two-role setup

| Role | Model | Purpose in PPI |
|---|---|---|
| **System under evaluation** | `claude-haiku-4-5-20251001` | Generates candidate SQL for each question |
| **LLM judge (proxy)** | `claude-sonnet-4-6` | Scores all predictions cheaply — but may be biased |

### Concept mapping

| PPI concept | Instantiation in this notebook |
|---|---|
| Proxy labels $\tilde{Y}_i$ | LLM judge binary verdict (0 = wrong, 1 = correct) |
| Human-grade labels $Y_i$ | Human annotation subset |
| Strata | Spider database domains (`db_id`) |
| Target metric $\mu$ | Fraction of correctly generated SQL queries |

## Section 1 — Load the dataset

The dataset was pre-built by the preparation scripts in `spider_case_study/scripts/`.
It contains 1 000 Spider examples with:
- the natural language question and gold SQL
- the predicted SQL from `claude-haiku-4-5-20251001`
- a binary correctness verdict from the LLM judge (`llm_judge_label`)
- a binary correctness verdict from the human annotator (`human_label`)

> **Note:** make sure to run the notebook from `spider_case_study/notebooks/`
> or adjust `DATASET_PATH` if your working directory differs.

In [ ]:
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from numpy.typing import NDArray

from glide.estimators import StratifiedClassicalMeanEstimator, StratifiedPPIMeanEstimator
from glide.samplers import StratifiedSampler

# ── Constants ──────────────────────────────────────────────────────────────
DATASET_PATH = Path("../data/spider_ppi_dataset.parquet")
CONFIDENCE_LEVEL = 0.95
RANDOM_SEED = 42
BUDGET = 200  # annotation budget

# Color palette
C_JUDGE = "#E74C3C"  # LLM judge / naive
C_HUMAN = "#2E86AB"  # classical / human-only
C_GLIDE = "#27AE60"  # stratified PPI++

In [ ]:
df = pd.read_parquet(DATASET_PATH)

print(f"Total examples    : {len(df):,}")
print(f"Databases (strata): {df['db_id'].nunique()}")
print(f"LLM judge rate    : {df['llm_judge_label'].mean():.1%}")
print(f"Human label rate  : {df['human_label'].mean():.1%}")

In [ ]:
display_cols = ["db_id", "question", "predicted_sql", "llm_judge_label", "human_label"]
df[display_cols].head(8)

## Section 2 — Stratified sampling

We annotate only a **budget** of examples, not the full 1 000.
Using a uniform random sample would waste budget on databases where the model always succeeds or always fails
(low variance — not informative). **Neyman allocation** does better: it concentrates budget on databases
where the proxy has higher variance, maximising statistical efficiency.

GLIDE's `StratifiedSampler` handles this automatically. It returns two arrays:
- `pi`: the sampling probability assigned to each example
- `xi`: a binary indicator — 1 if the example was selected for annotation, 0 otherwise

In [ ]:
y_proxy: NDArray = df["llm_judge_label"].to_numpy().astype(float)
groups: NDArray = df["db_id"].to_numpy()

pi, xi = StratifiedSampler().sample(
    y_proxy=y_proxy,
    groups=groups,
    budget=BUDGET,
    strategy="neyman",
    random_seed=RANDOM_SEED,
)

print(f"Annotation budget  : {BUDGET}")
print(f"Examples selected  : {int(xi.sum())}")

In [ ]:
allocation = (
    df.assign(sampled=xi)
    .groupby("db_id")
    .agg(
        total=("example_id", "count"),
        sampled=("sampled", "sum"),
        proxy_variance=("llm_judge_label", "var"),
    )
    .assign(sampling_rate=lambda d: d["sampled"] / d["total"])
    .sort_values("proxy_variance", ascending=False)
)

print("Per-database allocation (sorted by proxy variance, descending):")
print(allocation.to_string(float_format=lambda x: f"{x:.3f}"))

Databases with higher proxy variance (the judge gives a mix of 0s and 1s — it is uncertain)
receive a higher sampling rate. Databases where the judge is always 0 or always 1 receive fewer annotations
because there is little to learn from labelling them.

## Section 3 — Reveal human labels

In a real annotation workflow, we would now send the selected examples to human annotators.
Here we simulate this by **unmasking** the human labels only for the sampled rows.

GLIDE's convention: `y_true` is a numpy array where:
- sampled rows carry their actual label (0 or 1)
- unsampled rows carry `np.nan`

This masking is the only change needed to go from the full dataset to the labelled-subset view.

In [ ]:
human_label_all: NDArray = df["human_label"].to_numpy().astype(float)
y_true: NDArray = np.where(xi == 1, human_label_all, np.nan)

n_labeled = int((~np.isnan(y_true)).sum())
n_unlabeled = int(np.isnan(y_true).sum())

print(f"Labeled examples   : {n_labeled}")
print(f"Unlabeled examples : {n_unlabeled}")
print()
print(f"Observed human accuracy on labeled subset : {np.nanmean(y_true):.1%}")
print(f"LLM judge accuracy on full set            : {np.mean(y_proxy):.1%}")
print()
print(f"Estimated judge bias (rectifier estimate) : {np.nanmean(y_true) - np.nanmean(y_proxy[xi == 1]):.3f}")

## Section 4 — Three estimators compared

We compute the same accuracy estimate three ways:

1. **Naive** — take the raw mean of the LLM judge over all 1 000 examples.
   Fast and uses all data, but inherits the judge's bias.

2. **Classical** — use only the 200 human-labeled examples.
   Unbiased, but the confidence interval is wide because $n = 200$.

3. **Stratified PPI++** — combine both. GLIDE corrects the judge's bias using the labeled subset
   and builds a confidence interval that leverages all 1 000 proxy values.
   Unbiased *and* narrow CI.

In [ ]:
naive_result = StratifiedClassicalMeanEstimator().estimate(
    y_proxy,
    groups=groups,
    metric_name="SQL Correctness (Naive)",
    confidence_level=CONFIDENCE_LEVEL,
)

classical_result = StratifiedClassicalMeanEstimator().estimate(
    y_true,
    groups=groups,
    metric_name="SQL Correctness (Stratified Classical)",
    confidence_level=CONFIDENCE_LEVEL,
)

sppi_result = StratifiedPPIMeanEstimator().estimate(
    y_true=y_true,
    y_proxy=y_proxy,
    groups=groups,
    metric_name="SQL Correctness (Stratified PPI++)",
    confidence_level=CONFIDENCE_LEVEL,
    power_tuning=True,
)

In [ ]:
sep = "-" * 74
print(f"{'Estimator':<40} {'Estimate':>8}   {'95% CI':>20}   {'Width':>7}")
print(sep)

for result in [naive_result, classical_result, sppi_result]:
    ci = result.confidence_interval
    label = result.metric_name
    print(f"{label:<40} {result.mean:>8.1%}   [{ci.lower_bound:.1%}, {ci.upper_bound:.1%}]   {ci.width:>7.1%}")

## Section 5 — Visualisation

The chart below summarises the three estimates with their 95% confidence intervals.
The dashed line marks the oracle mean (computed from all 1 000 human labels — unavailable in practice).

Key observations:
- The **naive estimate** may sit away from the oracle line, reflecting the judge's systematic bias.
- The **classical estimate** is centred near the oracle but has a wide interval because $n = 200$.
- The **Stratified PPI++ estimate** is both centred near the oracle *and* has the narrowest interval.

In [ ]:
methods: List[str] = ["Stratified LLM-as-Judge", "Stratified Classical", "Stratified PPI++"]
means: List[float] = [naive_result.mean, classical_result.mean, sppi_result.mean]
half_widths: List[float] = [
    naive_result.width / 2,
    classical_result.width / 2,
    sppi_result.width / 2,
]
colors: List[str] = [C_JUDGE, C_HUMAN, C_GLIDE]

fig = go.Figure()

for method, mean, half_width, color in zip(methods, means, half_widths, colors):
    fig.add_trace(
        go.Bar(
            name=method,
            x=[method],
            y=[mean],
            error_y=dict(type="data", array=[half_width], visible=True, color="black", thickness=2.5, width=10),
            marker_color=color,
            marker_line_color="white",
            marker_line_width=1.5,
        )
    )

fig.update_layout(
    title=dict(
        text="SQL Correctness Estimates with 95% Confidence Intervals",
        font_size=16,
    ),
    yaxis=dict(
        title="Estimated Correctness Rate",
        tickformat=".0%",
        range=[0, 1.05],
    ),
    xaxis_title="Estimator",
    showlegend=False,
    template="plotly_white",
    width=700,
    height=500,
    bargap=0.35,
)

fig.show()

## Section 6 — Effective sample size

How many human labels would the stratified classical estimator need to match PPI++'s precision?

Since confidence interval width scales as $1/\sqrt{n}$, we can read the answer directly from the width ratio:

$$n_{\text{eff}} = n_{\text{labeled}} \times \left(\frac{w_{\text{classical}}}{w_{\text{PPI++}}}\right)^2$$

A ratio greater than 1 means PPI++ is more precise than the classical estimator for the same annotation budget — the proxy labels are doing useful work.

In [ ]:
width_ratio = classical_result.width / sppi_result.width
effective_sample_size = int(n_labeled * width_ratio**2)
annotation_saving = 1 - n_labeled / effective_sample_size

print(f"Classical CI width  : {classical_result.width:.4f}")
print(f"PPI++ CI width      : {sppi_result.width:.4f}")
print(f"Width ratio         : {width_ratio:.2f}x")
print()
print(f"Annotation budget used      : {n_labeled}")
print(f"Effective sample size       : {effective_sample_size}")
print(f"Annotation cost reduction   : {annotation_saving:.0%}")

## Summary

| | Naive (LLM Judge) | Stratified Classical | Stratified PPI++ |
|---|---|---|---|
| **Data used** | 1 000 proxy labels | 200 human labels | 200 human + 1 000 proxy |
| **Unbiased** | Only if judge is perfect | Yes | Yes |
| **CI width** | Narrow (but misleading if biased) | Wide | Narrow |
| **Stratification** | No | Yes (Neyman allocation) | Yes (Neyman allocation) |

### Three takeaways

1. **LLM judges can be systematically biased.** A narrow CI from the naive estimator does not mean an accurate estimate — it just means the judge is confident in its (potentially wrong) verdict.

2. **Stratified PPI++ reduces annotation cost.** The effective sample size printed above quantifies this: PPI++ achieves the precision of a much larger classical study with only a fraction of the human labels.

3. **Neyman allocation concentrates budget wisely.** Databases where the model is inconsistent (high proxy variance) receive more annotations, extracting more statistical information per annotation euro.